In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import drive
if os.path.exists('/content/inputs.json.gz'):
    D = '/content'
else:
    drive.mount('/content/drive')
    D = '/content/drive/MyDrive/temporal-ood-ddi'
os.makedirs(f'{D}/runs/preds', exist_ok=True)
print(D)

In [ ]:
!mkdir -p /content/w && tar xzf {D}/ddiood_code.tgz -C /content/w 2>/dev/null; cp {D}/inputs.json.gz /content/w/
!pip -q install rank_bm25 bitsandbytes accelerate

In [ ]:
import hashlib
p = '/content/w/ddiood/gpu_jobs.py'
s = open(p).read()
a = '        J += [job("bert", "krag", "biomedbert", s, 2, chars=400, epochs=10)]\n'
s = s.replace(a, a + a.replace('s, 2, ', 's, 2, variant="contaminated", '), 1)
b = 'job("bert", "b2mixed", "biobert", priority=3)]'
s = s.replace(b, 'job("bert", "b2mixed", "biobert", priority=3), job("bert", "b1names", "biobert", priority=3)]', 1)
h = hashlib.sha256(s.encode()).hexdigest()
assert h == 'ab8abe15625ecd3cde973a24a56306bea4e12129ab62ea15a1fcadb13f267a32', h
with open(p, 'w') as f:
    f.write(s)
print('registry ready', h)

In [ ]:
JOBS = [
 "b2sentence-biobert-s42",
 "b2mixed-biobert-s42",
 "b1names-biobert-s42"
]

In [ ]:
%cd /content/w
!python -m ddiood.gpu_jobs --bundle inputs.json.gz --out {D}/runs --only {' '.join(JOBS)} 2>&1 | tee -a {D}/runs/E6_architecture_check.log | grep --line-buffered -E '^(start|done|skip)|Error|Traceback'

In [ ]:
import os
done = sorted(j for j in JOBS if os.path.exists(f'{D}/runs/preds/{j}.json'))
print(f'{len(done)}/{len(JOBS)} jobs finished')
print('missing:', [j for j in JOBS if j not in done])